In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project="my-dissertation-project-502620")

In [ ]:
def extract_discharge_summary(subject_id, hadm_id):
    """
    Extract the discharge summary for a hospital admission.

    Parameters
    ----------
    subject_id : int
        Patient identifier.
    hadm_id : int
        Hospital admission identifier.

    Returns
    -------
    pandas.DataFrame
        Discharge summary note.
    """

    query = """
    SELECT
        note_id,
        charttime,
        storetime,
        text
    FROM `physionet-data.mimiciv_note.discharge`

    WHERE
        subject_id = @subject_id
        AND hadm_id = @hadm_id

    ORDER BY charttime
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("subject_id", "INT64", subject_id),
            bigquery.ScalarQueryParameter("hadm_id", "INT64", hadm_id),
        ]
    )

    patient_discharge_summary = (
        client.query(query, job_config=job_config).to_dataframe()
    )
    if not patient_discharge_summary.empty:
        patient_discharge_summary["text"] = patient_discharge_summary["text"].str.replace(r"\n", "\n", regex=False)
    return patient_discharge_summary

In [ ]:
import re


def extract_physical_examination(text):
    """
    Extract physical examination findings that are safe for model input.

    The function:
    1. Extracts the Physical Examination section.
    2. Removes discharge/end-of-admission examination findings by
       truncating the extracted section at the first unsafe heading.
    3. Returns only the earlier physical examination information.
    """

    if not text:
        return ""

    # ---------------------------------------------------------
    # 1. Normalize whitespace
    # ---------------------------------------------------------
    text = text.replace("\n", " ")


    # ---------------------------------------------------------
    # 2. Possible headings that start the Physical Exam section
    # ---------------------------------------------------------
    pe_strings = [
        r"physical exam\s*:",
        r"physical examination\s*:",
        r"physical ___\s*:",
        r"pe\s*:",
        r"pe ___\s*:"
    ]


    # ---------------------------------------------------------
    # 3. Headings that indicate the end of the overall
    #    Physical Examination section
    # ---------------------------------------------------------
    terminal_pattern = (
        r"(?="
        r"pertinent results\s*:|"
        r"brief hospital course\s*:|"
        r"discharge diagnosis\s*:|"
        r"discharge medications?\s*:|"
        r"discharge disposition\s*:|"
        r"discharge instructions?\s*:|"
        r"follow[- ]?up\s*:|"
        r"$"
        r")"
    )


    # ---------------------------------------------------------
    # 4. Extract the complete Physical Examination section
    # ---------------------------------------------------------
    extracted = ""

    for pe_str in pe_strings:

        regex = re.compile(
            rf"{pe_str}(.*?){terminal_pattern}",
            re.IGNORECASE | re.DOTALL
        )

        match = regex.search(text)

        if match:
            extracted = match.group(1).strip()
            break


    if not extracted:
        return ""


    # ---------------------------------------------------------
    # 5. Identify discharge/end-of-admission examination
    #    headings observed in the selected cases.
    #
    #    Once one is encountered, everything after it is removed.
    # ---------------------------------------------------------
    unsafe_pe_pattern = re.compile(
        r"(?:"
        
        # Anything explicitly referring to discharge
        r"\bdischarge\b"
        
        r"|"

        # Expiration examination
        r"\bexpiration\b"

        r"|"

        # Final physical examination
        r"\bfinal\s+physical\s+exam(?:ination)?\b"

        r"|"

        # Prior-to-discharge examination
        r"\bprior\s+to\s+discharge\b"

        r")",
        re.IGNORECASE
    )


    unsafe_match = unsafe_pe_pattern.search(extracted)

    if unsafe_match:
        extracted = extracted[:unsafe_match.start()]


    # ---------------------------------------------------------
    # 6. Clean whitespace
    # ---------------------------------------------------------
    extracted = re.sub(r"\s+", " ", extracted)


    return extracted.strip()

In [ ]:
def extract_patient_physical_exam_subheadings(patient_physical_exam):
    """
    Extract candidate subsection headings from a physical examination
    section.

    Returns
    -------
    set
        Unique candidate headings found in the PE section.
    """

    if not patient_physical_exam:
        return set()

    subheadings = set()

    lines = patient_physical_exam.splitlines()

    for line in lines:

        line = line.strip()

        if not line:
            continue

        # Pattern 1:
        # Heading followed by a colon
        # e.g. GENERAL:, HEENT:, NEURO:, Vitals:
        match = re.match(
            r"^([A-Za-z][A-Za-z /_-]{1,50})\s*:",
            line
        )

        if match:
            subheadings.add(match.group(1).strip())

        # Pattern 2:
        # Physical-exam-specific headings without a colon
        # e.g. Admission physical exam
        #      Discharge Physical Exam
        match = re.match(
            r"^((?:admission|initial|discharge|"
            r"at discharge|on discharge|upon discharge)\s+"
            r"physical\s+exam(?:ination)?)\b",
            line,
            re.IGNORECASE
        )

        if match:
            subheadings.add(match.group(1).strip())

    return subheadings

In [ ]:
def inspect_discharge_pe_context(patient_physical_exam, context=500):
    """
    Print context surrounding discharge/end-of-admission
    physical examination headings.
    """

    if not patient_physical_exam:
        return

    unsafe_pattern = re.compile(
        r"(?:"
        r".{0,50}\bdischarge\b.{0,50}|"
        r".{0,50}\bexpiration\b.{0,50}|"
        r".{0,50}\bfinal\s+physical\s+exam\b.{0,50}"
        r")",
        re.IGNORECASE
    )

    for match in unsafe_pattern.finditer(patient_physical_exam):

        start = max(0, match.start() - context)
        end = min(len(patient_physical_exam), match.end() + context)

        print("-" * 80)
        print(patient_physical_exam[start:end])
        print("-" * 80)

In [ ]:
query = """
SELECT
    subject_id,
    hadm_id
FROM `my-dissertation-project-502620.Query_Results.final_100_evaluation_cases`
"""

selected_cases = client.query(query).to_dataframe()

final_subheading_set = set()

# for i, row in selected_cases.iterrows():

#     subject_id = row["subject_id"]
#     hadm_id = row["hadm_id"]

#     patient_discharge_summary = extract_discharge_summary(
#         subject_id,
#         hadm_id
#     )

#     if not patient_discharge_summary.empty:

#         full_text = "\n".join(
#             patient_discharge_summary["text"].astype(str)
#         )

#         # Exploratory/raw extraction:
#         # intentionally includes all PE subsections.
#         patient_physical_exam = extract_physical_examination(
#             full_text
#         )

#         temp = extract_patient_physical_exam_subheadings(
#             patient_physical_exam
#         )

#         final_subheading_set.update(temp)
for i, row in selected_cases.iterrows():

    subject_id = row["subject_id"]
    hadm_id = row["hadm_id"]

    patient_discharge_summary = extract_discharge_summary(
        subject_id,
        hadm_id
    )

    if not patient_discharge_summary.empty:

        full_text = "\n".join(
            patient_discharge_summary["text"].astype(str)
        )

        patient_physical_exam = extract_physical_examination(
            full_text
        )

        if patient_physical_exam:

            print("\n" + "=" * 100)
            print(f"CASE: subject_id={subject_id}, hadm_id={hadm_id}")
            print("=" * 100)

            inspect_discharge_pe_context(patient_physical_exam)

In [ ]:
final_subheading_set